# Causal dual-head training: XLM-R vs mBERT (CS6140 Group 5)

**Settings before running:** Accelerator = **GPU T4 x2**, Internet = **On** (downloads the two pretrained models).
Add the dataset `cs6140-phase1-v4` (the uploaded zip) as input.

Run with **Save Version → Save & Run All (Commit)** so it keeps running with the browser closed.
Everything is written to `/kaggle/working/runs` (kept as notebook output).

Rule: model choices (epoch, threshold) use **validation only**. Test is scored once at the end; do not tune on test numbers.

In [ ]:
import glob, os, subprocess
!nvidia-smi --query-gpu=name,memory.total --format=csv
ROOT = os.path.dirname(glob.glob('/kaggle/input/**/labels/train_labels.jsonl', recursive=True)[0]).rsplit('/labels', 1)[0]
DATA = ROOT + '/data'  if os.path.exists(ROOT + '/data/labels') else ROOT
CODE = glob.glob('/kaggle/input/**/streaming/train.py', recursive=True)[0].rsplit('/train.py', 1)[0]
print('DATA =', DATA); print('CODE =', CODE)
!cp -r {CODE} /kaggle/working/streaming
import transformers, torch; print('torch', torch.__version__, 'transformers', transformers.__version__)

## 1. Causal-mask leakage test

In [ ]:
!python /kaggle/working/streaming/tests/test_causal.py

## 2. Train both models in parallel (GPU 0: XLM-R, GPU 1: mBERT), then predict val + test

Same hyper-parameters for both: 3 epochs, batch 64, encoder lr 3e-5, λ_sw = λ_dur = 1.

In [ ]:
!bash /kaggle/working/streaming/run_all.sh {DATA} /kaggle/working/runs --epochs 3 --batch_size 64

## 3. Logs and results

In [ ]:
!tail -n 3 /kaggle/working/runs/xlmr.log /kaggle/working/runs/mbert.log
!cat /kaggle/working/runs/xlmr/val_log.csv /kaggle/working/runs/mbert/val_log.csv
!grep -h -A30 "Model:" /kaggle/working/runs/*_test_eval.txt

In [ ]:
# Keep outputs small: drop checkpoints from the saved output if space is tight (each ~1 GB).
!du -sh /kaggle/working/runs/*